In [1]:
import torch
from torch import nn
from torch.nn import functional as F

x = torch.arange(4)
torch.save(x, 'x-file')

In [2]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

我们可以存储一个张量列表，然后把它们读回内存。

In [3]:
y = torch.zeros(4)
torch.save([x, y],'x-files')
x2, y2 = torch.load('x-files')
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

我们甚至可以写入或读取从字符串映射到张量的字典。 当我们要读取或写入模型中的所有权重时，这很方便。

In [4]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

In [13]:
class MLP(nn.Module):
    def __init__(self, num_inputs, num_hiddens, num_outputs):
        super().__init__()
        self.hidden = nn.Linear(num_inputs, num_hiddens)
        self.output = nn.Linear(num_hiddens, num_outputs)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP(20, 256, 10)
X = torch.randn(size=(2, 20))
Y = net(X)

In [6]:
torch.save(net.state_dict(), 'mlp.params')

In [7]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [8]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

In [ ]:
torch.save({
    "epoch": epoch,
    "model_state_dict": net.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss_value
}, "checkpoint.pth")

In [9]:
import copy
old_net = nn.Sequential(
    nn.Linear(20, 64),  # 第0个模块
    nn.ReLU(),          # 第1个模块
    nn.Linear(64, 32),  # 第2个模块
    nn.ReLU(),          # 第3个模块
    nn.Linear(32, 10)   # 第4个模块
)
front = copy.deepcopy(old_net[:2])

new_net = nn.Sequential(
    front,
    nn.Linear(64, 5)
)

In [10]:
torch.save(net, "entire_model.pth")
net = torch.load(
    "entire_model.pth",
    weights_only=False
)

In [17]:
checkpoint = {
    "model_type": "MLP",
    "model_config": {
        "num_inputs": 20,
        "num_hiddens": 256,
        "num_outputs": 10
    },
    "model_state_dict": net.state_dict()
}

torch.save(checkpoint, "model_checkpoint.pth")

In [19]:
checkpoint = torch.load(
    "model_checkpoint.pth",
    weights_only=True,
    map_location="cpu"
)

config = checkpoint["model_config"]

net = MLP(
    num_inputs=config["num_inputs"],
    num_hiddens=config["num_hiddens"],
    num_outputs=config["num_outputs"]
)

net.load_state_dict(checkpoint["model_state_dict"])
net.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)